In [ ]:
# ============================================================
# CELL 0 – ENVIRONMENT SETUP
# ============================================================

import numpy as np
import pandas as pd
import os

if os.path.exists('/kaggle/input'):
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

try:
    import kagglehub
except ImportError:
    pass


In [ ]:
# ============================================================
# CELL 1 – INSTALL / IMPORT CORE LIBRARIES
# ============================================================

!pip install -q xgboost shap openpyxl

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

print("Environment ready.")


In [ ]:
# ============================================================
# STEP 5B – AUTOMATICALLY FIND land_acquisition_master.csv
# ============================================================

from pathlib import Path
import os

FILE_NAME = "land_acquisition_master.csv"

candidates = [
    Path(FILE_NAME),
    Path(".") / FILE_NAME,
    Path("..") / FILE_NAME,
]

DATA_PATH = None
for candidate in candidates:
    if candidate.is_file():
        DATA_PATH = str(candidate.resolve())
        break

if not DATA_PATH:
    search_dirs = [Path("."), Path("..")]
    if Path("/kaggle/input").exists():
        search_dirs.append(Path("/kaggle/input"))
    if Path("/content").exists():
        search_dirs.append(Path("/content"))

    for search_dir in search_dirs:
        try:
            matches = list(search_dir.rglob(FILE_NAME))
            if matches:
                DATA_PATH = str(matches[0].resolve())
                break
        except Exception:
            continue

if not DATA_PATH:
    raise FileNotFoundError(f"Could not find '{FILE_NAME}'. Looked in current directory ({Path.cwd()}) and subdirectories.")

print("Dataset found!")
print("Exact path:")
print(DATA_PATH)


In [ ]:
# ============================================================
# STEP 5C – LOAD DATASET
# ============================================================

import pandas as pd

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


In [ ]:
# ============================================================
# STEP 5D – SHOW ALL COLUMNS
# ============================================================

print("Total columns:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")


In [ ]:
# ============================================================
# STEP 6 – BASIC DATASET INFORMATION
# ============================================================

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDataset information:")
df.info()


In [ ]:
# ============================================================
# STEP 7 – PROJECT / SNAPSHOT CHECK
# ============================================================

print("Unique projects:", df["project_id"].nunique())
print("Unique snapshots:", df["snapshot_id"].nunique())

snapshots_per_project = df.groupby("project_id")["snapshot_id"].nunique()

print("\nSnapshots per project:")
print(snapshots_per_project.value_counts().sort_index())


In [ ]:
# ============================================================
# STEP 8 – DUPLICATE CHECK
# ============================================================

duplicate_rows = df.duplicated().sum()
duplicate_project_snapshots = df.duplicated(subset=["project_id", "snapshot_id"]).sum()

print("Duplicate complete rows:", duplicate_rows)
print("Duplicate project + snapshot:", duplicate_project_snapshots)


In [ ]:
# ============================================================
# STEP 9 – MISSING VALUE ANALYSIS
# ============================================================

missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean().mul(100).round(2)
})

missing_report = missing_report.sort_values("missing_percentage", ascending=False)
display(missing_report[missing_report["missing_count"] > 0])


In [ ]:
# ============================================================
# STEP 10 – PERCENTAGE RANGE VALIDATION
# ============================================================

percentage_columns = [
    "land_acquired_pct", "land_pending_pct", "private_land_pct",
    "government_land_pct", "forest_land_pct", "compensation_pending_pct",
    "document_completion_pct", "rr_completion_pct", "families_relocated_pct",
    "possession_pct", "row_available_pct", "encumbrance_free_pct",
    "stakeholder_response_rate"
]

for column in percentage_columns:
    if column in df.columns:
        invalid_count = ((df[column] < 0) | (df[column] > 100)).sum()
        print(f"{column:35} invalid = {invalid_count}")


In [ ]:
# ============================================================
# STEP 11 – LAND CONSISTENCY
# ============================================================

land_total = df["land_acquired_pct"] + df["land_pending_pct"]
land_difference = (land_total - 100).abs()

print("Maximum difference:", land_difference.max())
print("Average difference:", land_difference.mean())


In [ ]:
# ============================================================
# STEP 12 – COMPENSATION CONSISTENCY
# ============================================================

calculated_pending = df["compensation_awarded_amount"] - df["compensation_paid_amount"]
difference = (calculated_pending - df["compensation_pending_amount"]).abs()

print("Maximum difference:", difference.max())
print("Average difference:", difference.mean())


In [ ]:
# ============================================================
# STEP 13 – TARGET DISTRIBUTION
# ============================================================

print("Counts:")
print(df["delayed_gt_90_days"].value_counts())

print("\nPercentages:")
print(df["delayed_gt_90_days"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# ============================================================
# STEP 14 – VERIFY TARGET DEFINITION
# ============================================================

expected_delayed = (df["actual_completion_delay_days"] > 90).astype(int)
mismatches = (expected_delayed != df["delayed_gt_90_days"]).sum()

print("Target mismatches:", mismatches)


In [ ]:
# ============================================================
# STEP 15 – DELAY SEVERITY CHECK
# ============================================================

print(df["delay_severity"].value_counts())


In [ ]:
# ============================================================
# STEP 15B – VERIFY SEVERITY LOGIC
# ============================================================

def calculate_severity(days):
    if days <= 0: return "on_time"
    elif days <= 30: return "minor"
    elif days <= 90: return "moderate"
    elif days <= 180: return "severe"
    else: return "critical"

expected_severity = df["actual_completion_delay_days"].apply(calculate_severity)
severity_mismatches = (expected_severity != df["delay_severity"]).sum()
print("Severity mismatches:", severity_mismatches)


In [ ]:
# ============================================================
# STEP 16 – LIFECYCLE STAGE DISTRIBUTION
# ============================================================

stage_distribution = df["acquisition_stage"].value_counts()
print(stage_distribution)


In [ ]:
# ============================================================
# STEP 16B – INSPECT ONE PROJECT'S TIMELINE
# ============================================================

sample_project_id = df["project_id"].iloc[0]
project_timeline = df[df["project_id"] == sample_project_id].sort_values("snapshot_id")

display(project_timeline[["project_id", "snapshot_id", "acquisition_stage", "land_acquired_pct", "possession_pct"]])


In [ ]:
# ============================================================
# STEP 17 – SNAPSHOT PROGRESSION
# ============================================================

display(project_timeline[["snapshot_id", "compensation_paid_amount", "document_completion_pct"]])


In [ ]:
# ============================================================
# STEP 18 – LEAKAGE COLUMNS
# ============================================================

LEAKAGE_COLUMNS = [
    "actual_completion_delay_days",
    "delayed_gt_90_days",
    "delay_severity",
    "most_likely_delay_stage"
]

print("Columns excluded from model:")
for col in LEAKAGE_COLUMNS:
    print(" -", col)


In [ ]:
# ============================================================
# STEP 19B – AUTOMATED DOMAIN FEATURE ENGINEERING (B.L.A.S.T.)
# ============================================================

# 1. Financial Ratios
df["cost_per_hectare"] = df["project_cost"] / (df["land_required_hectares"] + 1e-5)
df["cost_per_km"] = df["project_cost"] / (df["project_length_km"] + 1e-5)
df["compensation_to_cost_ratio"] = df["compensation_awarded_amount"] / (df["project_cost"] + 1e-5)
df["compensation_disbursement_rate"] = df["compensation_paid_amount"] / (df["compensation_awarded_amount"] + 1e-5)
df["pending_comp_to_cost"] = df["compensation_pending_amount"] / (df["project_cost"] + 1e-5)

# 2. Legal & Dispute Density
df["total_litigation_cases"] = df["legal_case_count"] + df["court_case_count"] + df["arbitration_case_count"] + df["ownership_dispute_count"]
df["litigation_per_family"] = (df["legal_case_count"] + df["court_case_count"]) / (df["affected_families"] + 1)
df["litigation_per_km"] = df["total_litigation_cases"] / (df["project_length_km"] + 1e-5)
df["public_friction_index"] = (df["public_objection_count"] + df["unresolved_grievances"] + df["rr_grievances"]) / (df["affected_families"] + 1)

# 3. Lifecycle Velocity & Temporal Momentum
df["land_acquisition_rate"] = df["land_acquired_pct"] / (df["days_since_notification"] + 1)
df["rr_velocity"] = df["rr_completion_pct"] / (df["days_since_notification"] + 1)
df["stage_stagnation_ratio"] = df["days_in_current_stage"] / (df["days_since_notification"] + 1)
df["administrative_lag_share"] = (df["notification_delay_days"] + df["approval_delay_days"] + df["survey_delay_days"]) / (df["days_since_notification"] + 1)

# Longitudinal delta features within each project
df = df.sort_values(["project_id", "snapshot_id"]).reset_index(drop=True)
df["delta_land_acquired_pct"] = df.groupby("project_id")["land_acquired_pct"].diff().fillna(0)

# 4. SHAP-Guided Interaction Terms
df["stakeholder_friction_x_comp_pending"] = (100 - df["stakeholder_response_rate"]) * df["compensation_pending_pct"] / 100
df["possession_deficit"] = df["land_acquired_pct"] - df["possession_pct"]

ENGINEERED_FEATURES = [
    "cost_per_hectare", "cost_per_km", "compensation_to_cost_ratio",
    "compensation_disbursement_rate", "pending_comp_to_cost",
    "total_litigation_cases", "litigation_per_family", "litigation_per_km", "public_friction_index",
    "land_acquisition_rate", "rr_velocity", "stage_stagnation_ratio", "administrative_lag_share",
    "delta_land_acquired_pct", "stakeholder_friction_x_comp_pending", "possession_deficit"
]

print(f"Engineered {len(ENGINEERED_FEATURES)} domain-specific features across Financial, Legal, Velocity, and Interaction terms.")


In [ ]:
# ============================================================
# STEP 20 – FINAL FEATURES FOR MODEL 1 (EXPANDED HIGH-PERFORMANCE)
# ============================================================

BASE_FEATURES = [
    "state", "district", "project_type",
    "project_cost", "project_length_km", "land_required_hectares",
    "land_acquired_pct", "land_pending_pct", "private_land_pct", "government_land_pct", "forest_land_pct",
    "affected_families", "affected_landowners", "vulnerable_households",
    "compensation_awarded_amount", "compensation_paid_amount", "compensation_pending_amount",
    "compensation_pending_pct", "compensation_dispute_count", "average_compensation_delay_days",
    "legal_case_count", "court_case_count", "arbitration_case_count", "ownership_dispute_count",
    "notification_delay_days", "approval_delay_days", "survey_delay_days",
    "document_completion_pct", "interdepartmental_pending_count",
    "rr_required", "rr_completion_pct", "families_relocated_pct", "rr_grievances",
    "possession_pct", "row_available_pct", "encumbrance_free_pct",
    "public_objection_count", "unresolved_grievances", "stakeholder_response_rate",
    "district_avg_resolution_days", "agency_avg_delay_days", "previous_project_delay_rate",
    "days_since_notification", "days_since_last_update", "days_in_current_stage",
    "acquisition_stage"
]

FEATURE_COLUMNS = BASE_FEATURES + ENGINEERED_FEATURES

missing_features = [col for col in FEATURE_COLUMNS if col not in df.columns]
print("Missing expected features:", missing_features)
print(f"Total model features: {len(FEATURE_COLUMNS)} (Base: {len(BASE_FEATURES)} + Engineered: {len(ENGINEERED_FEATURES)})")


In [ ]:
# ============================================================
# STEP 21 – FEATURE MATRIX + TARGET
# ============================================================

X = df[FEATURE_COLUMNS].copy()
y = df["delayed_gt_90_days"].astype(int).copy()
groups = df["project_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups:", groups.nunique())


In [ ]:
# ============================================================
# STEP 23 – GROUP-BASED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx]
y_train_val = y.iloc[train_val_idx]
groups_train_val = groups.iloc[train_val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]
groups_test = groups.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss_val.split(X_train_val, y_train_val, groups=groups_train_val))

X_train = X_train_val.iloc[train_idx]
y_train = y_train_val.iloc[train_idx]
groups_train = groups_train_val.iloc[train_idx]

X_val = X_train_val.iloc[val_idx]
y_val = y_train_val.iloc[val_idx]
groups_val = groups_train_val.iloc[val_idx]

print("TRAIN\nRows:", len(X_train), "\nProjects:", groups_train.nunique())
print("\nVALIDATION\nRows:", len(X_val), "\nProjects:", groups_val.nunique())
print("\nTEST\nRows:", len(X_test), "\nProjects:", groups_test.nunique())


In [ ]:
# ============================================================
# STEP 24 – VERIFY ZERO PROJECT OVERLAP
# ============================================================

train_projects = set(groups_train)
validation_projects = set(groups_val)
test_projects = set(groups_test)

print("Train & Validation:", len(train_projects & validation_projects))
print("Train & Test:", len(train_projects & test_projects))
print("Validation & Test:", len(validation_projects & test_projects))


In [ ]:
# ============================================================
# STEP 26 – IDENTIFY FEATURE TYPES
# ============================================================

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\nCategorical:\n", categorical_features)


In [ ]:
# ============================================================
# STEP 27 – PREPROCESSING PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessor created successfully.")


In [ ]:
# ============================================================
# STEP 28 – LOGISTIC REGRESSION BASELINE
# ============================================================

from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

baseline_model.fit(X_train, y_train)
print("Logistic Regression baseline trained.")


In [ ]:
# ============================================================
# STEP 29 – BASELINE EVALUATION
# ============================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

baseline_val_prob = baseline_model.predict_proba(X_val)[:, 1]
baseline_val_pred = (baseline_val_prob >= 0.5).astype(int)

baseline_metrics = {
    "Accuracy": accuracy_score(y_val, baseline_val_pred),
    "Precision": precision_score(y_val, baseline_val_pred, zero_division=0),
    "Recall": recall_score(y_val, baseline_val_pred, zero_division=0),
    "F1": f1_score(y_val, baseline_val_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, baseline_val_prob),
    "PR-AUC": average_precision_score(y_val, baseline_val_prob)
}

print("LOGISTIC REGRESSION\n" + "=" * 50)
for k, v in baseline_metrics.items():
    print(f"{k:12}: {v:.4f}")

print("\nConfusion Matrix:\n", confusion_matrix(y_val, baseline_val_pred))


In [ ]:
# ============================================================
# STEP 30 – XGBOOST CHAMPION MODEL
# ============================================================

from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

xgb_model.fit(X_train, y_train)
print("XGBoost training completed successfully.")


In [ ]:
# ============================================================
# STEP 31 – XGBOOST VALIDATION RESULTS
# ============================================================

xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
xgb_val_pred = (xgb_val_prob >= 0.5).astype(int)

xgb_metrics = {
    "Accuracy": accuracy_score(y_val, xgb_val_pred),
    "Precision": precision_score(y_val, xgb_val_pred, zero_division=0),
    "Recall": recall_score(y_val, xgb_val_pred, zero_division=0),
    "F1": f1_score(y_val, xgb_val_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, xgb_val_prob),
    "PR-AUC": average_precision_score(y_val, xgb_val_prob)
}

print("XGBOOST\n" + "=" * 50)
for k, v in xgb_metrics.items():
    print(f"{k:12}: {v:.4f}")

print("\nConfusion Matrix:\n", confusion_matrix(y_val, xgb_val_pred))


In [ ]:
# ============================================================
# STEP 32 – MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **baseline_metrics},
    {"Model": "XGBoost", **xgb_metrics}
])

display(comparison)


In [ ]:
# ============================================================
# STEP 33 – THRESHOLD ANALYSIS
# ============================================================

threshold_results = []
for threshold in np.arange(0.20, 0.81, 0.05):
    predictions = (xgb_val_prob >= threshold).astype(int)
    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "f1": f1_score(y_val, predictions, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)


In [ ]:
best_threshold = threshold_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
print("Best validation threshold:", best_threshold)


In [ ]:
# ============================================================
# STEP 34 – RISK LEVEL
# ============================================================

def get_risk_level(probability: float) -> str:
    if probability < 0.40: return "LOW"
    elif probability < 0.60: return "MODERATE"
    elif probability < 0.80: return "HIGH"
    else: return "CRITICAL"


In [ ]:
# ============================================================
# STEP 35 – EXTRACT TRAINED PREPROCESSOR + XGBOOST
# ============================================================

trained_preprocessor = xgb_model.named_steps["preprocessor"]
trained_classifier = xgb_model.named_steps["classifier"]

X_val_transformed = trained_preprocessor.transform(X_val)
feature_names = trained_preprocessor.get_feature_names_out()

print("Transformed feature count:", len(feature_names))


In [ ]:
# ============================================================
# STEP 36 – SHAP
# ============================================================

import shap

explainer = shap.TreeExplainer(trained_classifier)
sample_size = min(1000, X_val_transformed.shape[0])
sample_matrix = X_val_transformed[:sample_size]

shap_values = explainer(sample_matrix)
print("SHAP calculation completed.")


In [ ]:
# ============================================================
# STEP 37 – SHAP SUMMARY
# ============================================================

shap.summary_plot(shap_values, features=sample_matrix, feature_names=feature_names, show=True)


In [ ]:
# ============================================================
# STEP 38 – SINGLE PROJECT PREDICTION
# ============================================================

example_index = 0
example_project = X_test.iloc[[example_index]]
probability = xgb_model.predict_proba(example_project)[0, 1]

print("Delay probability:", round(probability * 100, 2), "%")
print("Risk:", get_risk_level(probability))


In [ ]:
# ============================================================
# STEP 39 – INDIVIDUAL PROJECT EXPLANATION
# ============================================================

example_transformed = trained_preprocessor.transform(example_project)
example_explanation = explainer(example_transformed)
example_values = example_explanation.values[0]

individual_explanation = pd.DataFrame({
    "feature": feature_names,
    "shap_value": example_values
})

individual_explanation["absolute_impact"] = individual_explanation["shap_value"].abs()
top_factors = individual_explanation.sort_values("absolute_impact", ascending=False).head(10)

display(top_factors[["feature", "shap_value"]])


In [ ]:
# ============================================================
# STEP 40A – DIAGNOSTIC
# ============================================================

if "X_test_transformed" not in locals():
    X_test_transformed = trained_preprocessor.transform(X_test)

print("Test rows:", len(X_test))
print("Test columns:", len(X_test.columns))
print("Transformed rows:", X_test_transformed.shape[0])
print("Transformed features:", X_test_transformed.shape[1])
print("Number of XGBoost trees:", trained_classifier.n_estimators)
print("Max depth:", trained_classifier.max_depth)


In [ ]:
# ============================================================
# STEP 40 – FAST FINAL TEST EVALUATION
# ============================================================

import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

print("Transforming test data...")
start_time = time.time()
X_test_transformed = trained_preprocessor.transform(X_test)
print(f"Transformation completed in {time.time() - start_time:.2f} seconds")

print("\nRunning XGBoost inference...")
start_time = time.time()
test_probability = trained_classifier.predict_proba(X_test_transformed)[:, 1]
print(f"Inference completed in {time.time() - start_time:.2f} seconds")

test_prediction = (test_probability >= best_threshold).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(y_test, test_prediction),
    "Precision": precision_score(y_test, test_prediction, zero_division=0),
    "Recall": recall_score(y_test, test_prediction, zero_division=0),
    "F1": f1_score(y_test, test_prediction, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_probability),
    "PR-AUC": average_precision_score(y_test, test_probability)
}

print("\n" + "=" * 60)
print("FINAL TEST RESULTS (UNSEEN INDEPENDENT 1,200 PROJECTS)")
print("=" * 60)
for metric, value in final_metrics.items():
    print(f"{metric:12}: {value:.4f}")

print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
print(confusion_matrix(y_test, test_prediction))
